# 두 YOLO 모델 test 평가

두 모델은 이미 학습됐다. val 기준으로 YOLO11n (960)을 최종 모델로 미리 선택했다. test 결과는 최종 보고용이며 모델 선택이나 재학습에 사용하지 않는다.


In [ ]:
!pip -q install ultralytics==8.4.208
import json, zipfile, yaml, torch, ultralytics
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0), "Ultralytics:", ultralytics.__version__)
drive.mount("/content/drive")


## 데이터와 가중치 확인


In [ ]:
DATA_ZIP = Path("/content/drive/MyDrive/construction-ppe-5class.zip")
DATA_PARENT = Path("/content/ppe_project_data")
DATA_ROOT = DATA_PARENT / "construction-ppe-5class"
assert DATA_ZIP.is_file(), f"데이터 ZIP 없음: {DATA_ZIP}"
if not (DATA_ROOT / "split_manifest.csv").is_file():
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATA_ZIP) as archive:
        assert archive.testzip() is None, "ZIP이 손상되었습니다."
        archive.extractall(DATA_PARENT)
config = yaml.safe_load((DATA_ROOT / "data.yaml").read_text())
config["path"] = str(DATA_ROOT)
DATA_YAML = DATA_ROOT / "data_colab.yaml"
DATA_YAML.write_text(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))
assert len(list((DATA_ROOT / "images/test").glob("*.jpg"))) == 100
ROOT = Path("/content/drive/MyDrive/ppe-project")
WEIGHTS = {"yolo11n": ROOT / "runs/baseline_yolo11n_seed42/weights/best.pt",
           "yolo26n": ROOT / "runs/proposed_yolo26n_seed42/weights/best.pt",
           "yolo11n_960": ROOT / "runs/improved_yolo11n_img960_seed42/weights/best.pt"}
for name, path in WEIGHTS.items():
    assert path.is_file(), f"가중치 없음: {path}"
    print(name, path)


## test 평가
세 모델에 동일한 test 100장을 적용한다. 입력 크기는 각 모델의 검증 설정을 따른다 (960 또는 640). 결과표와 혼동행렬은 Drive에 저장된다.


In [ ]:
OUT = ROOT / "test_evaluation"
for name in WEIGHTS:
    if (OUT / name).exists():
        raise FileExistsError(f"기존 평가 결과가 있습니다: {OUT / name}")
OUT.mkdir(parents=True, exist_ok=True)
summary = {}
for name, weight in WEIGHTS.items():
    result = YOLO(str(weight)).val(data=str(DATA_YAML), split="test", imgsz=960 if name == "yolo11n_960" else 640, batch=8, device=0, plots=True, project=str(OUT), name=name, exist_ok=False)
    box = result.box
    summary[name] = {"precision": float(box.mp), "recall": float(box.mr), "mAP50": float(box.map50), "mAP50_95": float(box.map), "classes": {str(result.names[int(c)]): {"precision": float(box.p[i]), "recall": float(box.r[i]), "AP50": float(box.ap50[i]), "AP50_95": float(box.ap[i])} for i, c in enumerate(box.ap_class_index)}}
    (OUT / f"{name}_metrics.json").write_text(json.dumps(summary[name], ensure_ascii=False, indent=2))
(OUT / "comparison_metrics.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("저장 위치:", OUT)
